# 02 · Querying Beehive Data

**Goal: master the query call and the table it returns.**

Notebook `01` used `query` to discover nodes and metrics. This notebook slows
down and takes the call apart completely: the time arguments, the filter, and the
shape of the DataFrame that comes back. Everything after this notebook is built
on these three things, so it is worth being thorough here.

By the end you will be able to phrase almost any question about a single node or
metric as a query, read every column of the result, and save it for reuse.

In [ ]:
import sage_data_client
import pandas as pd

In [ ]:
# Import the SageTools helper package (github.com/mpapka/SageTools).
#
# SageTools is a convenience layer built on top of the official sage_data_client.
# It is optional. Every lab in this course also shows the underlying
# sage_data_client call directly, so the helper is never a hard dependency.
#
# The try/except means the notebook runs whether or not the package is installed:
# if it is missing, haveSageTools becomes False and the notebook falls back to
# the direct calls it already teaches. To install it, see notebook 00.
try:
    import sage
    haveSageTools = True
except ImportError:
    haveSageTools = False

print("SageTools helper package available:", haveSageTools)

## 1. The call, in full

```python
sage_data_client.query(
    start,          # REQUIRED, where the window begins
    end=None,       # optional, where it ends, defaults to now
    head=None,      # optional, first N matching rows
    tail=None,      # optional, last N matching rows
    filter=None,    # optional dict, narrows what comes back
)
```

There are only five arguments and you have met all of them. This notebook goes
deep on `start`/`end` (time) and `filter`, because those are where nearly all
real questions live.

## 2. Time arguments in depth

`start` and `end` accept two spellings, and mixing them is fine.

### Relative offsets

A minus sign followed by a number and a unit means "this long ago, measured from
now."

| offset | meaning |
| --- | --- |
| `-30m` | thirty minutes ago |
| `-6h` | six hours ago |
| `-7d` | seven days ago |
| `-2w` | two weeks ago |

Relative offsets are convenient for "recent" questions and for cells you re-run,
because they always mean "recent relative to right now."

### Absolute timestamps

An absolute time is an ISO 8601 string in UTC, ending in `Z`:
`2024-06-01T00:00:00Z`. Use absolute times when you need a fixed, repeatable
window, for example "all of June 2024," which should not drift as the days pass.

### Two rules that save you grief

- **Times are UTC.** The Beehive stores and returns timestamps in Coordinated
  Universal Time, not your local zone. If a reading looks six hours off from what
  you expect, this is almost always why. Convert to local time only for display,
  and label it when you do.
- **`start` is required, `end` defaults to now.** If you omit `end`, the window
  runs up to the present moment.

In [ ]:
# Demonstrate both spellings. These two windows are different questions.
# A: the last six hours, relative to whenever you run this cell.
relativeDf = sage_data_client.query(
    start="-6h",
    filter={"name": "env.temperature", "vsn": "W06C"},
)
print("relative window rows:", len(relativeDf))

# B: a fixed absolute day, the same every time you run it.
absoluteDf = sage_data_client.query(
    start="2024-06-01T00:00:00Z",
    end="2024-06-02T00:00:00Z",
    filter={"name": "env.temperature", "vsn": "W06C"},
)
print("absolute window rows:", len(absoluteDf))

## 3. The filter dictionary in depth

The `filter` dict is how you narrow a query. Each key you add makes the result
smaller and the query faster. The keys you will use:

- **`name`**: the metric, for example `env.temperature`. Omit it during discovery
  to see every metric.
- **`vsn`**: the node call sign, for example `W06C`.
- **`node`**: a lower level node identifier. `vsn` is what you will normally use.
- **`sensor`**: restrict to one physical sensor on a node.
- **`plugin`**: restrict to measurements produced by a specific plugin.

### Filters can match by pattern

Filter values are treated as text patterns, so you can match loosely with regular
expression syntax. For example `{"plugin": ".*mobotix.*"}` matches any plugin
whose name contains "mobotix." This is useful when you know part of a name but not
all of it. For exact metric and node filters you will usually just give the full
string.

In [ ]:
# The focused query you will run most often: one metric, one node, one window.
tempDf = sage_data_client.query(
    start="-3h",
    filter={"name": "env.temperature", "vsn": "W06C"},
)
print("records:", len(tempDf))
tempDf.head()

## 4. The returned DataFrame, column by column

Every query returns a pandas DataFrame in the same schema. Knowing each column is
what lets you go from a raw table to an answer.

| column | meaning | notes |
| --- | --- | --- |
| `timestamp` | when the reading was taken | UTC, parse with `pd.to_datetime` |
| `name` | the metric name | matches your filter |
| `value` | the measured number | temperature is in Celsius |
| `meta.vsn` | node call sign | which node produced the row |
| `meta.sensor` | physical sensor | a node may carry several |
| `meta.host` | on-node host identifier | useful for sensor auditing |

Other `meta.*` columns can appear depending on the metric, including
`meta.lat` and `meta.lon` for location. Temperature `value` is in **degrees
Celsius**. Always confirm a metric's unit before interpreting its number; humidity
is a percentage, pressure is in its own unit, and so on.

In [ ]:
# Inspect the schema, the column dtypes, and a numeric summary of the values.
if not tempDf.empty:
    print("columns:")
    for columnName in tempDf.columns:
        print("  ", columnName)
    print("\nvalue column summary (Celsius):")
    print(tempDf["value"].describe())
else:
    print("No rows returned. Try a longer window or a different node.")

## 5. A reusable fetch function

In practice you wrap the query so you do not repeat yourself. The function below
fetches one metric for one node, parses the timestamp into a real datetime, and,
when the metric is temperature, adds a Fahrenheit column for convenience. It
optionally saves the result to CSV, which is the handoff format to later
notebooks.

Read it top to bottom: build the filter, run the query, bail out cleanly on an
empty result, tidy the frame, and optionally save.

In [ ]:
def fetchMetric(nodeVsn, metricName="env.temperature", start="-6h", end="now", saveTo=None):
    """
    Fetch one metric for one node and return a tidy DataFrame.

    Args:
        nodeVsn: node call sign, for example "W06C".
        metricName: the metric to pull, default env.temperature.
        start, end: the time window, relative or absolute.
        saveTo: optional path. If given, the result is written to CSV.

    Returns:
        A DataFrame sorted by time. For temperature it also carries a valueF
        (Fahrenheit) column. Empty if nothing matched.
    """
    resultDf = sage_data_client.query(
        start=start,
        end=end,
        filter={"name": metricName, "vsn": nodeVsn},
    )
    if resultDf.empty:
        print(f"No {metricName} for {nodeVsn} in [{start}, {end}].")
        return resultDf

    resultDf["timestamp"] = pd.to_datetime(resultDf["timestamp"])
    resultDf = resultDf.sort_values("timestamp").reset_index(drop=True)

    if metricName == "env.temperature":
        resultDf["valueF"] = resultDf["value"] * 9 / 5 + 32

    if saveTo:
        resultDf.to_csv(saveTo, index=False)
        print(f"saved {len(resultDf)} rows to {saveTo}")

    return resultDf


recentDf = fetchMetric("W06C", start="-6h")
recentDf.head()

In [ ]:
# The SageTools equivalent adds friendly error hints and CSV export.
if haveSageTools:
    from sage.beehive.extraction.query import runQuery
    runQuery(start="-6h", name="env.temperature", vsn="W06C")

## 6. Saving results for later notebooks

CSV is the bridge between notebooks. Save a query result here and you can reopen
it in the analysis and visualization labs without querying again. The analysis
tools recover a node's call sign from the filename, so a name like
`chicago_temp_W06C.csv` is a good convention.

In [ ]:
savedDf = fetchMetric("W06C", metricName="env.temperature", start="-1d",
                      saveTo="chicago_temp_W06C.csv")

## 7. Troubleshooting queries

- **A 500 server error.** The query was too broad, so the Beehive gave up while
  assembling it. Narrow the time window, add a `name` filter, or add a `vsn`
  filter. Notebook `03` shows how to pull long windows safely by chunking them.
- **Zero rows.** The query worked but nothing matched. Confirm the node actually
  reports that metric (notebook `01`), widen the window, or check the spelling of
  the metric name.
- **Timestamps look shifted.** They are UTC. Convert to local time only for
  display, and label the zone.
- **Values look wrong by a constant factor or offset.** Check units. Temperature
  is Celsius. Converting to Fahrenheit uses `value * 9/5 + 32`.

## 8. Exercises

1. Pull `env.relative_humidity` for a live node over the last 3 hours using
   `fetchMetric`. Confirm there is no `valueF` column, and explain why the
   function does not add one for humidity.
2. Query the same absolute day twice, hours apart. Do you get identical row
   counts? What does that tell you about querying fixed windows versus relative
   ones?
   *Hint: an absolute window does not move; a `-6h` window does.*
3. Fetch temperature from two nodes into two frames and compute each mean. State
   the two numbers, then, separately, offer one possible reason one node runs
   warmer.
4. Add an argument to `fetchMetric` that limits the result to the last N rows
   using the query's `tail` parameter. When would you want that?
   *Hint: `tail` belongs inside the `query` call, not applied afterward.*

## Further reading

- Sage query reference (filters, regex, time format): https://sagecontinuum.org/docs/reference-guides/dev-quick-reference
- pandas `DataFrame`: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html
- pandas `to_datetime` for parsing timestamps: https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html
- ISO 8601, the absolute time format the API accepts: https://en.wikipedia.org/wiki/ISO_8601